# 环节 08 · 逃逸攻击面演示

纯 Python 标准库，零依赖。把"沙箱会不会被打穿"变成可穷举的判定：

1. 攻击面清单 + 每条路径的阻断条件；
2. 输入一份配置，输出可用逃逸路径；
3. 加固前后对比。

In [ ]:
# §1 攻击面清单：每条路径 = 一个"闸门条件"（全部满足才可逃逸）
PATHS = {
    "特权容器":          lambda c: c["privileged"],
    "挂 docker.sock":    lambda c: "docker.sock" in c["mounts"],
    "多余 CAP_SYS_ADMIN": lambda c: "CAP_SYS_ADMIN" in c["caps"],
    "cgroup v1 release_agent": lambda c: c["cgroup_version"] == 1 and not c["cgroup_ro"],
    "可写 /proc/sys":    lambda c: not c["proc_ro"],
    "宿主路径挂载":      lambda c: any(m in c["mounts"] for m in ["/home", "/var/run", "/root"]),
    "设备节点暴露":      lambda c: any(m.startswith("/dev/sd") for m in c["mounts"]),
    "可访问宿主网络":    lambda c: not c["net_isolated"],
    "可读云元数据":      lambda c: (not c["net_isolated"]) and not c["block_link_local"],
    "无资源配额":        lambda c: not c["has_quota"],
}

BAD = dict(privileged=True, mounts=["/var/run", "/home", "docker.sock"],
           caps=["CAP_SYS_ADMIN", "CAP_NET_ADMIN"], cgroup_version=1, cgroup_ro=False,
           proc_ro=False, net_isolated=False, block_link_local=False, has_quota=False)
GOOD = dict(privileged=False, mounts=[], caps=[], cgroup_version=2, cgroup_ro=True,
            proc_ro=True, net_isolated=True, block_link_local=True, has_quota=True)


def audit(cfg):
    return [name for name, cond in PATHS.items() if cond(cfg)]


print("配置 A（默认 Docker 风格、图省事）可用逃逸路径:")
for p in audit(BAD):
    print("  ✗", p)
print()
print("配置 B（加固后）可用逃逸路径:", audit(GOOD) or "无")

In [ ]:
# §2 逐项加固：每一刀砍掉哪些路径
FIXES = [
    ("去掉 --privileged",        lambda c: {**c, "privileged": False}),
    ("去掉 CAP_SYS_ADMIN",       lambda c: {**c, "caps": [x for x in c["caps"] if x != "CAP_SYS_ADMIN"]}),
    ("不挂 docker.sock",         lambda c: {**c, "mounts": [m for m in c["mounts"] if m != "docker.sock"]}),
    ("切换 cgroup v2 并只读",    lambda c: {**c, "cgroup_version": 2, "cgroup_ro": True}),
    ("/proc/sys 只读",           lambda c: {**c, "proc_ro": True}),
    ("不挂宿主路径",             lambda c: {**c, "mounts": []}),
    ("隔离网络",                 lambda c: {**c, "net_isolated": True}),
    ("拒绝链路本地",             lambda c: {**c, "block_link_local": True}),
    ("加资源配额",               lambda c: {**c, "has_quota": True}),
]

cfg = BAD
print(f"起始可用路径: {len(audit(cfg))}")
for label, fix in FIXES:
    before = len(audit(cfg))
    cfg = fix(cfg)
    after = len(audit(cfg))
    print(f"  {label:<22} {before} → {after}")
print()
print("剩余:", audit(cfg) or "无")
print()
print("结论：绝大多数逃逸是配置问题，逐个关闸门即可清零 —— 不需要内核补丁")

In [ ]:
# §3 配置评审清单自动打分
CHECKLIST = {
    "不用 --privileged": lambda c: not c["privileged"],
    "cap-drop ALL": lambda c: len(c["caps"]) == 0,
    "不挂 docker.sock": lambda c: "docker.sock" not in c["mounts"],
    "不挂宿主路径": lambda c: not any(m in c["mounts"] for m in ["/home", "/var/run", "/root"]),
    "cgroup v2 且只读": lambda c: c["cgroup_version"] == 2 and c["cgroup_ro"],
    "/proc/sys 只读": lambda c: c["proc_ro"],
    "网络默认隔离": lambda c: c["net_isolated"],
    "拒绝链路本地": lambda c: c["block_link_local"],
    "有资源配额": lambda c: c["has_quota"],
}


def score(cfg):
    passed = [k for k, f in CHECKLIST.items() if f(cfg)]
    return passed, len(passed) / len(CHECKLIST)


for label, cfg in [("A 图省事", BAD), ("B 加固后", GOOD)]:
    passed, ratio = score(cfg)
    print(f"{label}: {ratio:.0%} ({len(passed)}/{len(CHECKLIST)})  未通过={[k for k in CHECKLIST if k not in passed] or '无'}")
print()
print("任何一项『未通过』都应在威胁模型里显式记录为接受的风险，而不是默认忽略")

## §4 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | 容器逃逸最常见原因？ | 配置错误（特权、挂 socket、多余 cap）而非 0day |
| 2 | 为什么 cgroup 文件系统要只读？ | v1 的 `release_agent` 可让内核在宿主执行程序 |
| 3 | 面对内核 0day 的三件事？ | 缩小攻击面 + 升级隔离档 + 假设会被攻破（凭证兜底） |
| 4 | "有 0day 所以沙箱没用"对吗？ | 不对；混淆"降低概率"与"降低后果"，且配置层能挡多数真实攻击 |
| 5 | 加固验收必须包含什么？ | 主动越界测试（读宿主文件/连私网/fork 炸弹/写 /proc/sys） |
| 6 | 依赖安装脚本为什么要进沙箱？ | pip/npm postinstall 是真实攻击面 |

**相关长文**：[环节08-逃逸攻击面与加固详解.md](./环节08-逃逸攻击面与加固详解.md)